In [1]:
%reload_ext autoreload
%autoreload 2

In [4]:
import numpy as np
from topoflow.flow.paths import GaussianConditionalProbabilityPath, DiscreteConditionalProbabilityPath
from topoflow.flow.schedulers import LinearAlpha, LinearBeta
from topoflow.utils.visualization import visualize_data_rdkit, visualize_data, visualize_trajectory, visualize_data_matplotlib
from topoflow.flow.flow_models_discrete import FlowModel as FlowModelDiscrete
from topoflow.flow.flow_models_ligand import FlowModel
from topoflow.data.dataloader_adj import DataLoaderSampleable
from topoflow.data.datasets import CrossDocked
from topoflow.flow.schedulers import LinearAlpha, LinearBeta, CosineAlpha, CosineBeta

In [6]:
batch_size = 32
data_dir = "../data/CrossDocked"
train_dataset = CrossDocked(
    root=data_dir,
    split="train",
)
val_dataset = CrossDocked(root=data_dir, split="val")

alpha = CosineAlpha()
beta = CosineBeta()

p_data = DataLoaderSampleable(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
)


distribution_hist = np.load("../data/CrossDocked/processed/size_distribution_full.npy", allow_pickle=True)
path = GaussianConditionalProbabilityPath(p_data=p_data, alpha=alpha, beta=beta, distribution_hist=distribution_hist)
discrete_path = DiscreteConditionalProbabilityPath(p_data=p_data, alpha=alpha, n_classes=11)
checkpoint_path = "../checkpoints/runs_cluster/10_03_199e_gat_discrete.ckpt"

#Load the model from the checkpoint
# model = FlowModelDiscrete.load_from_checkpoint(checkpoint_path, path=path, discrete_path=discrete_path, norm_path="../data/CrossDocked/processed/normalization_params_full.npz")
model = FlowModel.load_from_checkpoint(checkpoint_path, path=path, norm_path="../data/CrossDocked/processed/normalization_params_full.npz")

Loaded dataset from ../data/CrossDocked/processed/train_crossdocked_pyg.pt.
Loaded SMILES file from ../data/CrossDocked/processed/train_crossdocked_pyg_smiles.txt
Dataset loaded successfully with 946 samples.
Loaded dataset from ../data/CrossDocked/processed/val_crossdocked_pyg.pt.
Loaded SMILES file from ../data/CrossDocked/processed/val_crossdocked_pyg_smiles.txt
Dataset loaded successfully with 244 samples.
Entropy of n_nodes: H[N] 9.08344554901123
Entropy of n_nodes: H[N] 8.75682258605957


In [7]:
res_traj = model.sample_with_trajectory(1, n_steps=100, force_connected=False)
visualize_trajectory(res_traj, image_path="outputs/trajectory/mol_traj.png", gif_path="outputs/trajectory/mol_traj.gif", plot_every=2)

Visualizing trajectory: 100%|██████████| 100/100 [00:33<00:00,  2.98it/s]


In [ ]:
res = model.sample(1, n_steps=20)

Seed set to 42


In [9]:
visualize_data(res, image_path="outputs/generated_molecules_pymol.png", session_path="outputs/generated_molecules_session.pse")

In [8]:
visualize_data_rdkit(res, image_path="outputs/generated_molecules.png", mols_per_row=2, sub_img_size=(1200, 1200))

Image saved to: outputs/generated_molecules.png


[16:46:19] Both bonds on one end of an atropisomer are on the same side - atoms are: 17 25


In [47]:
distances, other_metrics = model.test_distribution(
                test_distribution=model.val_distribution,
                metrics=[
                    "qed",
                    "logp",
                    "sas",
                ],  # TODO: add lipinski, but also in the val_dist
                atom_decoder=model.atom_decoder,
            )

Sampling molecules: 100%|██████████████████████████████████████████| 2/2 [00:20<00:00, 10.33s/batch]


In [49]:
model.normalization_params

{'mean': array([ 7.7809190e-08, -9.0558565e-08,  5.2221260e-08], dtype=float32),
 'std': array([5.8403964, 5.7269087, 5.8340626], dtype=float32)}

In [48]:
other_metrics

{'validity': tensor(1.), 'connectivity': tensor(0.), 'uniqueness': tensor(0.)}

In [ ]:
from torch_geometric.data import Data
from topoflow.utils.metrics import compute_connectivity
from topoflow.utils.molecules import data_to_rdmol

final_mol = res_traj[-1]
final_rdmol = data_to_rdmol(final_mol, atom_decoder=model.atom_decoder, sanitize=False)

connected = compute_connectivity(final_rdmol)[0]
print(connected)

[]


In [ ]:
res_traj = model._test_integration(n_steps=80)
visualize_trajectory(res_traj, image_path="outputs/trajectory/mol_traj.png", gif_path="outputs/trajectory/mol_traj.gif", plot_every=2)